# Student Notebook: PyROOT Quick Start for Testbeam Data

In this notebook you will open a real testbeam `.root` file, inspect a `TTree`,
make histograms from detector channels, and use scaler counters to estimate the pion
content of the beam.

Work through the TODO cells in order. The notebook gives hints, but it intentionally
does not include the final commands or interpretation answers.

## Logbooks

Please keep these logbooks open while working through the notebook:

- [August logbook](https://codimd.web.cern.ch/xpQJaJnvQk64tcIXiIYtWA)
- [June logbook](https://codimd.web.cern.ch/yB8Ihfl4QGGRWXDTChaglQ#20260613)



## Part 1 -- Everything from a single run file

We'll start with just **one** file: a plain run file, named after the run number (e.g. `1781661387.root`). This is the direct output of our DAQ system -- no offline processing has touched it yet. It contains raw, per-event data stored in a `TTree`, not ready-made histograms.

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["figure.dpi"] = 110

DATA_DIR = Path(".")
RUN_FILE = DATA_DIR / "convertedToROOT" / "1781661387.root"

# If this notebook is inside StudentsNotebooks or VolunteersNotebooks,
# the data folder is one level above the notebook.
if not RUN_FILE.exists():
    DATA_DIR = Path("..")
    RUN_FILE = DATA_DIR / "convertedToROOT" / "1781661387.root"

# TODO 1:
# Open the ROOT file with ROOT.TFile.Open.
# Then print the opened filename and list the top-level objects in the file.
f = ROOT.TFile.Open(str(RUN_FILE))
print("Opened:", f.GetName())
f.ls()


`ROOT.TFile.Open` is the PyROOT equivalent of Python's built-in `open()`, but for
`.root` files. `ls()` should show `RAWdata` and `RECOdata`, both of type `TTree`.

**Question 1:** Which tree should you use if you want raw, per-event detector values?
Why is a `TTree` similar to a table?


In [ ]:
# TODO 2:
# Fetch the raw-data tree from the ROOT file.
# Then print the number of entries and the first few branch names.

tree = f.Get("...")  # replace ... with the tree name

print("Number of events (entries):", tree.GetEntries())
print()
print("A few of the branches:")
for branch in list(tree.GetListOfBranches())[:8]:
    print(" ", branch.GetName())


### Making your own histogram straight from the tree

PyROOT can turn any branch into a histogram with:

`tree.Draw("branch_name>>histogram_name(n_bins, x_min, x_max)")`

**Question 2:** Choose a QDC channel, for example `QDC0_ch2`. What do you think the
three numbers in `(150, 0, 350)` control?


In [ ]:
# TODO 3:
# Fill a histogram from QDC0_ch2.
# Use 150 bins between 0 and 350, and name the histogram h_ch2.

tree.Draw("...>>...(150, 0, 350)")
h_ch2 = ROOT.gDirectory.Get("...")

print("Entries: ", h_ch2.GetEntries())
print("Mean:    ", h_ch2.GetMean())
print("Std dev: ", h_ch2.GetStdDev())


`ROOT.gDirectory` is ROOT's current in-memory directory. `tree.Draw(...)` creates
the histogram there, and `.Get("histogram_name")` fetches it back.

**Question 3:** Before running the next cell, predict where most entries will appear:
near low ADC values, spread uniformly, or mostly at high ADC values?


In [ ]:
canvas = ROOT.TCanvas("canvas", "raw ADC", 700, 450)
h_ch2.SetLineColor(ROOT.kAzure + 2)
h_ch2.SetLineWidth(2)
h_ch2.Draw("HIST")
canvas.SetLogy()
canvas.SaveAs("qdc_ch2_raw.png")

Image("qdc_ch2_raw.png")

**Observe and discuss:**

1. Where is the tallest part of the ADC spectrum?
2. Do you see a tail toward higher ADC values?
3. What might the low-ADC peak represent physically?
4. Try another channel, such as `QDC0_ch0` through `QDC0_ch15`. Do all channels look
   similar, or are some clearly different?


### Pulling the histogram into NumPy for your own plots

Now read the ROOT histogram bins into NumPy arrays and redraw the same distribution
with `matplotlib`.

**Question 4:** Why might you want the bin contents in NumPy instead of only drawing
with ROOT?


In [ ]:
# TODO 4:
# Extract the bin edges and bin contents from h_ch2.
# Hint: ROOT histogram bin numbers start at 1 for the first visible bin.

n_bins = h_ch2.GetNbinsX()
edges = np.array([...])
counts = np.array([...])

plt.stairs(counts, edges, fill=True, color="steelblue")
plt.xlabel("QDC0_ch2 raw ADC counts")
plt.ylabel("entries")
plt.yscale("log")
plt.title("Same histogram, redrawn with matplotlib from PyROOT bin contents")
plt.show()


### Reading raw values event-by-event, and subtracting them

`tree.Draw` is great for quick histograms, but sometimes you need actual per-event
numbers in Python. Here we use scaler channels: `Scaler0_ch0` through `Scaler0_ch15`.

A **scaler** is a hardware counter. It increments whenever a signal arrives and does
not reset during the run. Each event stores the current cumulative count.

Important channels from the electronics diagram:

| channel | signal | meaning |
|---|---|---|
| `Scaler0_ch0` | Disc. C0 | Cherenkov C0 discriminator |
| `Scaler0_ch1` | Disc. C1 | Cherenkov C1 discriminator |
| `Scaler0_ch12` | EVTTRG | triggered and recorded events |
| `Scaler0_ch15` | 1 kHz clock | free-running timing reference |

The June electronics diagram labels C0 as the Cherenkov counter that can fire for
electrons, muons, and pions, while C1 is below the pion threshold and should mostly
fire for electrons and muons.

**Question 5:** With that mapping, which subtraction would estimate pion-like counts:
`C0 - C1` or `C1 - C0`? What would it mean if the run data appears to show the
opposite ordering?


In [ ]:
N_EVENTS = 20000

scaler_C0 = np.zeros(N_EVENTS)
scaler_C1 = np.zeros(N_EVENTS)
scaler_trg = np.zeros(N_EVENTS)
scaler_clk = np.zeros(N_EVENTS)

# TODO 5:
# Loop over the first N_EVENTS entries.
# For each event, load the entry and store Scaler0_ch0, Scaler0_ch1,
# Scaler0_ch12, and Scaler0_ch15 in the arrays above.

for i in range(N_EVENTS):
    ...

print("First 10 readings of C0 (ch0):    ", scaler_C0[:10])
print("First 10 readings of C1 (ch1):    ", scaler_C1[:10])
print("First 10 readings of EVTTRG (ch12):", scaler_trg[:10])


`tree.GetEntry(i)` loads event number `i` into memory; after that, every branch is available as a plain attribute, e.g. `tree.Scaler0_ch0`. This is the simplest (if not the fastest) way to loop over a `TTree` event by event in PyROOT.

Notice all these numbers only ever go up. Let's plot the raw, cumulative C0 and C1 counts together.

For this +3 GeV run, both XCET radiators were CO2:

| detector | pressure | number of events |
|---|---:|---:|
| XCET44 | 2.8 bar | 2137 |
| XCET48 | 2.2 bar | 1192 |

Useful CO2 threshold pressures at +3 GeV are approximately:

**e-: 3.22e-5 bar, muon: 1.378 bar, pion: 2.405 bar, kaon: 30.36 bar, proton: 106.1 bar**

**Question:** which XCET pressure is above the pion threshold, and which one is below it? Based on that, which detector should count electrons + muons + pions, and which should mostly count electrons + muons?


In [ ]:
plt.plot(scaler_C0, label="C0 discriminator", color="darkorange")
plt.plot(scaler_C1, label="C1 discriminator", color="steelblue")
plt.xlabel("event number")
plt.ylabel("cumulative scaler count")
plt.title("Raw scaler counts: both only ever count up")
plt.legend()
plt.show()


**Question 6:** Which curve grows faster, C0 or C1? Does that match the particle
types each Cherenkov counter is expected to count?


### Comparing C0 and C1: estimating pion-like counts

The electronics diagram says C0 is the pion-on Cherenkov counter and C1 is the
pion-off Cherenkov counter. With that mapping:

$$\text{pion-like count} \approx C0 - C1$$

Now use the first and last scaler values in the full run to estimate total counts.


In [ ]:
# TODO 6:
# Get the first and last entries in the tree.
# Store C0, C1, EVTTRG, and CLK for both entries.
# Then subtract first from last to get run totals.

n_all = tree.GetEntries()

tree.GetEntry(...)
C0_first, C1_first, TRG_first, CLK_first = ..., ..., ..., ...

tree.GetEntry(...)
C0_last, C1_last, TRG_last, CLK_last = ..., ..., ..., ...

C0_total = ...
C1_total = ...
TRG_total = ...
pion_like_count = ...  # use the subtraction implied by the electronics diagram

print(f"C0 total:                {C0_total}")
print(f"C1 total:                {C1_total}")
print(f"EVTTRG total (triggers): {TRG_total}")
print(f"pion-like scaler count:  {pion_like_count}")
print()
print(f"pion-like fraction of EVTTRG: {100 * pion_like_count / TRG_total:.2f} %")


**Question 7:** Compare the C0 and C1 totals with the EVTTRG total.

1. Are the raw Cherenkov discriminator counts smaller or larger than the triggered
   event count?
2. Does the larger Cherenkov scaler match the counter that the electronics diagram
   says should be pion-on?
3. If not, what should you check before drawing a physics conclusion?
4. Is a raw scaler subtraction necessarily a clean beam composition percentage?


In [ ]:
# TODO 7:
# Calculate the pion fraction relative to C1 instead of relative to EVTTRG.
# Also calculate the e+mu fraction relative to C1.

pion_fraction_of_C1 = ...
emu_fraction_of_C1 = ...

print(f"pion fraction of the beam (relative to C1): {pion_fraction_of_C1:.2f} %")
print(f"e+mu fraction of the beam (relative to C1):  {emu_fraction_of_C1:.2f} %")


**Final discussion question:** Which normalisation makes more physical sense for
the question "what fraction of the beam is pions": `/ EVTTRG` or `/ C1`?

Explain your answer using the trigger logic and the fact that C0 and C1 are raw,
ungated discriminator scalers. Check the logbook if you need to know exactly how
`EVTTRG` was defined for this run.
